<a href="https://colab.research.google.com/github/skminhazuddin/SixthSemesterProjects/blob/main/texttoimage01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================= FULL CLEAN RESET =================
!pip uninstall -y torch torchvision torchaudio xformers \
    diffusers transformers accelerate peft safetensors huggingface_hub \
    datasets gradio torchtune sentence-transformers

# ================= INSTALL CUDA PYTORCH FIRST =================
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# ================= RESTART (MANDATORY) =================
import os
os.kill(os.getpid(), 9)

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: diffusers 0.37.1
Uninstalling diffusers-0.37.1:
  Successfully uninstalled diffusers-0.37.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: safetensors 0.7.0
Uninstalling safetensors-0.7.0:
  Successfully uninstalled sa

In [2]:
# ================= INSTALL DIFFUSERS STACK =================
!pip install diffusers==0.25.0 \
            transformers==4.36.2 \
            accelerate==0.25.0 \
            peft==0.7.1 \
            safetensors \
            huggingface_hub==0.20.3

# ================= INSTALL XFORMERS (AFTER TORCH!) =================
#!pip install xformers==0.0.23.post1

In [3]:
!rm -rf ~/.cache/huggingface

In [4]:
from huggingface_hub import logout
from huggingface_hub import login
#logout()
login()

In [5]:
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download(
    repo_id="gsdf/Counterfeit-V2.5",
    filename="Counterfeit-V2.5.safetensors"
)
print(MODEL_PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Counterfeit-V2.5.safetensors:   0%|          | 0.00/7.70G [00:00<?, ?B/s]

/root/.cache/huggingface/hub/models--gsdf--Counterfeit-V2.5/snapshots/93c5412baf37cbfa23a3278f7b33b0328db581fb/Counterfeit-V2.5.safetensors


In [6]:
!rm -rf SixthSemesterProjects
!git clone https://github.com/skminhazuddin/SixthSemesterProjects.git -b main
#!ls /content
#!ls /content/SixthSemesterProjects
#!ls /content/SixthSemesterProjects/resource

Cloning into 'SixthSemesterProjects'...
remote: Enumerating objects: 1160, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 1160 (delta 34), reused 41 (delta 28), pack-reused 1080 (from 2)
Receiving objects: 100% (1160/1160), 647.44 MiB | 1.32 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Updating files: 100% (1065/1065), done.


In [9]:
# ================= IMPORTS =================
import os
import sqlite3
import warnings
from datetime import datetime
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.optim import AdamW
import torchvision.transforms as T

from diffusers import StableDiffusionXLPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model

warnings.filterwarnings("ignore")

# ================= CONFIG =================
MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0"

RESOLUTION = 256
BATCH_SIZE = 1
GRAD_ACCUM = 4

# ✅ LOWER LR (CRITICAL)
LEARNING_RATE = 1e-5
EPOCHS = 5

BASE_PATH = "/content/SixthSemesterProjects/resource"
DB_PATH = "/content/training_data.db"
CHAR_LORA_PATH = "/content/lora_character"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16

print("✅ Using device:", DEVICE)

# ================= DATABASE =================
class DatabaseManager:
    def __init__(self):
        print("📦 Initializing Database...")
        self.conn = sqlite3.connect(DB_PATH)
        self.create_tables()

    def create_tables(self):
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS images (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            image_path TEXT UNIQUE,
            description TEXT,
            tags TEXT,
            category TEXT,
            character TEXT,
            trained INTEGER DEFAULT 0
        )
        """)
        self.conn.commit()

    def insert_image(self, path, description, tags, category, character):
        cur = self.conn.cursor()
        cur.execute("SELECT id FROM images WHERE image_path=?", (path,))
        if cur.fetchone() is None:
            cur.execute("""
            INSERT INTO images (image_path, description, tags, category, character, trained)
            VALUES (?,?,?,?,?,0)
            """, (path, description, tags, category, character.lower()))
            self.conn.commit()

    def get_untrained_images(self, category):
        data = self.conn.execute("""
        SELECT id, image_path, description, tags, character
        FROM images WHERE trained=0 AND category=?
        """, (category,)).fetchall()

        print(f"📊 Found {len(data)} untrained images")
        return data

# ================= LOAD DATA =================
def load_images_to_db(db):
    print("📂 Scanning dataset folder...")

    for root, _, files in os.walk(BASE_PATH):
        for file in files:
            if file.lower().endswith((".png", ".jpg", ".jpeg")):
                img_path = os.path.join(root, file)
                txt_path = os.path.splitext(img_path)[0] + ".txt"

                desc = "bhonda character"

                if os.path.exists(txt_path):
                    with open(txt_path, "r", encoding="utf-8") as f:
                        content = f.read()
                        if "Description:" in content:
                            desc = content.split("Description:")[1].split("Tags:")[0].strip()

                db.insert_image(img_path, desc, "", "character", "")

# ================= DATASET =================
class ImageDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.transform = T.Compose([
            T.Resize((RESOLUTION, RESOLUTION)),
            T.ToTensor(),
            # ✅ FIXED normalization (RGB)
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        _, img_path, desc, _, _ = self.data[idx]

        img = Image.open(img_path).convert("RGB")

        return {
            "pixel_values": self.transform(img),
            "caption": desc.strip()
        }

# ================= TRAINER =================
class ModelTrainer:
    def __init__(self, dataset):
        print("🧠 Loading SDXL model...")

        self.pipe = StableDiffusionXLPipeline.from_pretrained(
            MODEL_NAME,
            torch_dtype=DTYPE,
            variant="fp16",
            use_safetensors=True
        )

        self.pipe.enable_model_cpu_offload()
        self.pipe.enable_attention_slicing()
        self.pipe.unet.enable_gradient_checkpointing()

        try:
            self.pipe.enable_xformers_memory_efficient_attention()
            print("✅ xformers enabled")
        except:
            print("⚠️ xformers not available")

        self.unet = self.pipe.unet
        self.vae = self.pipe.vae

        # ✅ CRITICAL FIX: VAE in fp32
        self.vae.to(dtype=torch.float32)

        self.tokenizer = self.pipe.tokenizer
        self.tokenizer_2 = self.pipe.tokenizer_2
        self.text_encoder = self.pipe.text_encoder
        self.text_encoder_2 = self.pipe.text_encoder_2

        print("🔧 Applying LoRA...")

        lora_config = LoraConfig(
            r=8,
            lora_alpha=8,
            target_modules=["to_q", "to_k", "to_v"],
            bias="none"
        )

        self.unet = get_peft_model(self.unet, lora_config)

        for name, param in self.unet.named_parameters():
            param.requires_grad = "lora" in name.lower()

        print("🧠 Trainable params:",
              sum(p.numel() for p in self.unet.parameters() if p.requires_grad))

        self.scheduler = DDPMScheduler.from_config(self.pipe.scheduler.config)
        self.dataset = dataset

    def train(self):
        loader = DataLoader(self.dataset, batch_size=BATCH_SIZE, shuffle=True)

        optimizer = AdamW(
            filter(lambda p: p.requires_grad, self.unet.parameters()),
            lr=LEARNING_RATE
        )

        print("🚀 Training started...")

        for epoch in range(EPOCHS):
            print(f"\n🔥 Epoch {epoch+1}/{EPOCHS}")

            for step, batch in enumerate(loader):

                pixel_values = batch["pixel_values"].to(DEVICE)

                # Tokenize
                inputs = self.tokenizer(
                    batch["caption"],
                    padding="max_length",
                    truncation=True,
                    max_length=77,
                    return_tensors="pt"
                ).to(DEVICE)

                inputs_2 = self.tokenizer_2(
                    batch["caption"],
                    padding="max_length",
                    truncation=True,
                    max_length=77,
                    return_tensors="pt"
                ).to(DEVICE)

                # ===== SAFE LATENT CREATION =====
                with torch.no_grad():
                    latents = self.vae.encode(
                        pixel_values.to(torch.float32)
                    ).latent_dist.sample()

                    latents = latents * 0.18215
                    latents = latents.to(DTYPE)

                    enc1 = self.text_encoder(inputs.input_ids, return_dict=True)
                    enc2 = self.text_encoder_2(inputs_2.input_ids, return_dict=True)

                    encoder_hidden_states = torch.cat(
                        [enc1.last_hidden_state, enc2.last_hidden_state], dim=-1
                    )

                    add_text_embeds = enc2.text_embeds

                add_time_ids = torch.tensor(
                    [[RESOLUTION, RESOLUTION, 0, 0, RESOLUTION, RESOLUTION]],
                    device=DEVICE,
                    dtype=torch.float32
                ).repeat(latents.shape[0], 1)

                noise = torch.randn_like(latents)
                timesteps = torch.randint(0, 1000, (latents.shape[0],), device=DEVICE)

                noisy_latents = self.scheduler.add_noise(latents, noise, timesteps)

                noise_pred = self.unet(
                    noisy_latents,
                    timesteps,
                    encoder_hidden_states=encoder_hidden_states,
                    added_cond_kwargs={
                        "text_embeds": add_text_embeds,
                        "time_ids": add_time_ids
                    }
                ).sample

                loss = F.mse_loss(noise_pred, noise)

                # ✅ NaN protection
                if torch.isnan(loss):
                    print("❌ NaN detected, skipping batch")
                    optimizer.zero_grad()
                    continue

                loss.backward()

                # ✅ Gradient clipping
                torch.nn.utils.clip_grad_norm_(self.unet.parameters(), 1.0)

                if (step + 1) % GRAD_ACCUM == 0:
                    optimizer.step()
                    optimizer.zero_grad()

                if step % 10 == 0:
                    print(f"Loss: {loss.item():.4f}")

        print("💾 Saving LoRA...")
        self.unet.save_pretrained(CHAR_LORA_PATH)
        print("✅ Saved!")

# ================= GENERATE =================
def generate_image():
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        variant="fp16"
    ).to(DEVICE)

    pipe.load_lora_weights(CHAR_LORA_PATH)

    prompt = input("Prompt: ")

    image = pipe(prompt, num_inference_steps=30).images[0]

    path = f"/content/output_{int(datetime.now().timestamp())}.png"
    image.save(path)

    print("✅ Saved:", path)

# ================= MAIN =================
def main():
    db = DatabaseManager()

    while True:
        print("\n1. Train\n2. Generate\n3. Exit")
        choice = input("Choice: ")

        if choice == "1":
            load_images_to_db(db)
            data = db.get_untrained_images("character")

            trainer = ModelTrainer(ImageDataset(data))
            trainer.train()

        elif choice == "2":
            generate_image()

        else:
            break

if __name__ == "__main__":
    main()

✅ Using device: cuda
📦 Initializing Database...

1. Train
2. Generate
3. Exit
Choice: 1
📂 Scanning dataset folder...
📊 Found 531 untrained images
🧠 Loading SDXL model...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

⚠️ xformers not available
🔧 Applying LoRA...
🧠 Trainable params: 8949760
🚀 Training started...

🔥 Epoch 1/5
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan


KeyboardInterrupt: 

In [ ]:
import sqlite3

conn = sqlite3.connect("/content/training_data.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM images;")

# Get column names
columns = [desc[0] for desc in cursor.description]

# Print header
print("\t".join(columns))

# Print rows
for row in cursor.fetchall():
    print("\t".join(str(x) for x in row))

id	image_path	description	tags	category	character	trained
1	/content/SixthSemesterProjects/resource/HandaBhondaComicFigures/Bhonda/Bhonda_09.jpg	Yellow face with exaggerated features, black hairstyle curl, thick eyebrows, intense expression with furrowed brows and frown. Bold outlines and simple shading.	cartoon, comic art, intense expression, stylized portrait, bold outlines
Character: Bhonda	character	bhonda	1
2	/content/SixthSemesterProjects/resource/HandaBhondaComicFigures/Bhonda/Bhonda_01.jpg	Bhonda face close-up portrait in classic Indian comic illustration style, round face with chubby cheeks, playful smiling expression, small eyes, short black hair with a small upward curl, vintage halftone print texture.	Bhonda, close-up portrait, Indian comic style, round face, chubby cheeks, playful smile, small eyes, short black hair, upward curl, vintage halftone texture, classic illustration.
Character: Bhonda	character	bhonda	1
3	/content/SixthSemesterProjects/resource/HandaBhondaComicFi

In [ ]:
#Check GPU memory
!nvidia-smi

Sun Apr 26 10:33:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----